# VideoDB Understanding API Preview

This notebook walks through the analyzer-based Understanding lifecycle using the SDK: create a run, inspect analyzers, wait for completion, fetch analyzer outputs, list runs, and optionally delete a run.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Video-DB/videodb-cookbook/blob/preview/guides/preview/understanding.ipynb)


## 1. Install dependencies

This preview notebook installs the SDK branch that contains Understanding helpers.


In [ ]:
!pip install -q --force-reinstall --no-cache-dir "git+https://github.com/Video-DB/videodb-python.git@feat/add-indexing-v2" python-dotenv


## 2. Connect to VideoDB

Set `VIDEO_DB_API_KEY` in your environment, or enter it when prompted.


In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

os.environ["VIDEO_DB_API_KEY"] = "" 

conn = connect()
print("Connected to VideoDB")

## 3. Pick a collection and video

Use your own collection and video IDs, or set `VIDEODB_COLLECTION_ID` and `VIDEODB_VIDEO_ID` in the environment.


In [ ]:
# Optional: list collections
conn.get_collections()

In [ ]:
COLLECTION_ID = os.getenv("VIDEODB_COLLECTION_ID", "c-d6bae500-e4c2-4547-b345-4881beaa7c73")
VIDEO_ID = os.getenv("VIDEODB_VIDEO_ID", "m-z-0197f422-5f72-78a1-985a-0baa4e88978a")

collection = conn.get_collection(COLLECTION_ID)
video = collection.get_video(VIDEO_ID)

print("Collection:", COLLECTION_ID)
print("Video:", video.id)

## 4. Create an Understanding run

Analyzers can use friendly names like `spoken_words`. The SDK maps this to the server analyzer and gives stable default output names: `transcript`, `objects`, and `scene`.


In [ ]:
understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "config": {"language": "en"},
        },
        {
            "type": "object_detection",
            "sampling": {"strategy": "interval", "every": 1},
            "config": {
                "labels": ["person", "vehicle", "animal", "product", "logo", "text"],
                "confidence_threshold": 0.35,
                "include_bounding_boxes": True,
            },
        },
        {
            "type": "vlm",
            "inputs": ["transcript", "objects"],
            "sampling": {"strategy": "uniform", "frame_count": 8},
            "config": {
                "model": "google/gemini-2.5-flash",
                "prompt": (
                    "Using the sampled frames, transcript, and detected objects, "
                    "describe what happens in this video scene by scene."
                ),
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding ID:", understanding.id)
print("Status:", understanding.status)
understanding.list_analyzers()

## 5. Refresh status and list analyzers

Use `refresh()` to pull the latest run/analyzer statuses.


In [ ]:
understanding.refresh()
print("Status:", understanding.status)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)

## 6. Wait until complete

`wait_until_complete()` polls until the run reaches a terminal status: `done`, `failed`, or `partial`.


In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Final status:", understanding.status)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.status)

## 7. Fetch analyzer outputs

Fetch output from an analyzer object with `get_analyzer(...).get_output()`.
The helper below prints a compact preview of scene-shaped analyzer outputs.


In [ ]:
def show_scenes_preview(analyzer_id, max_scenes=5):
    output = understanding.get_analyzer(analyzer_id).get_output()

    # Some API versions return {"scenes": [...]}; current versions may return the scenes list directly.
    scenes = output.get("scenes", output) if isinstance(output, dict) else output
    scenes = scenes or []

    print("analyzer :", analyzer_id)
    for i in scenes[:max_scenes]:
        print("start :", i.get("start"))
        print("end :", i.get("end"))
        print("data :", i.get("data"))
        print("=" * 10)


show_scenes_preview("transcript")


In [ ]:
show_scenes_preview("objects")


In [ ]:
show_scenes_preview("scene")


## 8. Work with an existing Understanding

You can fetch an existing run by ID and continue from there.


In [ ]:
same_understanding = video.get_understanding(understanding.id)
print(same_understanding)
same_understanding.list_analyzers()

## 9. List Understanding runs for this video


In [ ]:
understandings = video.list_understandings()
for item in understandings:
    print(item.id, item.status)

## 10. Optional: wait for one analyzer

Analyzer objects also support `refresh()` and `wait_until_complete()`.


In [ ]:
transcript_analyzer = understanding.get_analyzer("transcript")
transcript_analyzer.wait_until_complete(timeout=900, poll_interval=10)
print(transcript_analyzer.name, transcript_analyzer.status)

## 11. Optional: delete the Understanding

Only run this if you want to remove the Understanding records.


In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete. Set DELETE_UNDERSTANDING=True to delete this Understanding.")

## Endpoint mapping

The SDK wraps these video-scoped endpoints:

```text
POST   /video/<video_id>/understand
GET    /video/<video_id>/understand
GET    /video/<video_id>/understand/<understanding_id>
GET    /video/<video_id>/understand/<understanding_id>?analyzer=<name_or_id>
GET    /video/<video_id>/understand/<understanding_id>/analyzers/<name_or_id>/output
DELETE /video/<video_id>/understand/<understanding_id>
```
